# RDD Visual Analysis - Corrected Momentum Interpretation
Analyze each game separately with CORRECTED momentum interpretation:
- Strengthening trend: Momentum moving AWAY from zero (positive getting MORE positive, OR negative getting MORE negative)
- Weakening trend: Momentum moving TOWARD zero (positive getting LESS positive, OR negative getting LESS negative)
- Neutral: Starting near zero, analyzing post-treatment divergence/convergence

In [ ]:
import json
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## Load and Prepare Data

In [ ]:
# Load JSON files
with open('matches.json', 'r') as f:
    matches_df = pd.DataFrame(json.load(f))

with open('momentum.json', 'r') as f:
    momentum_df = pd.DataFrame(json.load(f))

with open('match_ids.json', 'r') as f:
    match_ids = json.load(f)

# Filter to match_ids
match_ids_str = [str(mid) for mid in match_ids]
matches_filtered = matches_df[matches_df['id'].astype(str).isin(match_ids_str)].copy()
momentum_filtered = momentum_df[momentum_df['id'].astype(str).isin(match_ids_str)].copy()

# Merge
merged_data = pd.merge(
    matches_filtered,
    momentum_filtered,
    on='id',
    suffixes=('_match', '_momentum'),
    how='inner'
)

print(f'Loaded {len(merged_data)} matches')
print(f'\nMomentum Metric Interpretation:')
print(f'  Positive momentum = Home team advantage')
print(f'  Negative momentum = Away team advantage')
print(f'  Zero momentum = Neutral/Balanced')
print(f'\nSlope Interpretation (CORRECTED):')
print(f'  STRENGTHENING = Momentum moving AWAY FROM ZERO')
print(f'    (positive slope when momentum > 0) OR (negative slope when momentum < 0)')
print(f'    = One team establishing/cementing advantage')
print(f'  WEAKENING = Momentum moving TOWARD ZERO')
print(f'    (negative slope when momentum > 0) OR (positive slope when momentum < 0)')
print(f'    = Advantage eroding, game becoming balanced')

## Per-Game Analysis: Extract Hydration Breaks and Calculate Trends

In [ ]:
def extract_hydration_windows(match_data, window_size=10):
    """
    For a single match, extract all hydration breaks and surrounding momentum data.
    
    CORRECTED Categorization:
    Strengthening: Momentum moving AWAY from zero
      - Pre-momentum > 0 AND pre_slope > +0.5 (positive getting more positive)
      - Pre-momentum < 0 AND pre_slope < -0.5 (negative getting more negative)
    Weakening: Momentum moving TOWARD zero
      - Pre-momentum > 0 AND pre_slope < -0.5 (positive getting less positive)
      - Pre-momentum < 0 AND pre_slope > +0.5 (negative getting less negative)
    Neutral: Pre-treatment momentum near zero
    """
    match_id = match_data['id']
    series = match_data['series']
    stoppages = match_data['stoppages']
    home_team = match_data['home_match']
    away_team = match_data['away_match']
    
    # Convert series to DataFrame
    if not isinstance(series, list):
        return []
    
    momentum_series = pd.DataFrame(series, columns=['minute', 'momentum'])
    
    # Find all hydration breaks
    hydration_breaks = []
    if isinstance(stoppages, list):
        for stoppage in stoppages:
            if len(stoppage) >= 2 and stoppage[1] == 'hydration':
                hydration_breaks.append({
                    'minute': stoppage[0],
                    'duration': stoppage[2] if len(stoppage) > 2 else None
                })
    
    if len(hydration_breaks) == 0:
        return []
    
    # Extract windows for each hydration break
    windows = []
    for hydration_break in hydration_breaks:
        hyd_minute = hydration_break['minute']
        
        # Define window bounds
        lower_bound = hyd_minute - window_size
        upper_bound = hyd_minute + window_size
        
        # Extract data in window
        window_data = momentum_series[
            (momentum_series['minute'] >= lower_bound) & 
            (momentum_series['minute'] <= upper_bound)
        ].copy()
        
        if len(window_data) < 3:  # Need at least 3 points for meaningful regression
            continue
        
        # Split into before/after
        before_data = window_data[window_data['minute'] < hyd_minute]
        after_data = window_data[window_data['minute'] >= hyd_minute]
        
        # Calculate slopes
        pre_slope = None
        post_slope = None
        pre_r2 = None
        post_r2 = None
        trend_change = None
        pre_avg_momentum = None
        post_avg_momentum = None
        pre_momentum_magnitude = None
        post_momentum_magnitude = None
        pre_trend_category = None
        
        if len(before_data) >= 2:
            z_before = np.polyfit(before_data['minute'], before_data['momentum'], 1)
            pre_slope = z_before[0]
            p_before = np.poly1d(z_before)
            ss_res = np.sum((before_data['momentum'] - p_before(before_data['minute']))**2)
            ss_tot = np.sum((before_data['momentum'] - before_data['momentum'].mean())**2)
            pre_r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            pre_avg_momentum = before_data['momentum'].mean()
            pre_momentum_magnitude = np.abs(pre_avg_momentum)
            
            # CORRECTED: Categorize pre-treatment trend based on direction AWAY FROM ZERO
            if -0.5 <= pre_slope <= 0.5:
                # Near-zero slope - trend is neutral
                pre_trend_category = 'neutral'
            elif pre_avg_momentum > 0:  # Positive momentum (Home team advantage)
                if pre_slope > 0.5:
                    pre_trend_category = 'strengthening'  # Positive getting more positive
                else:  # pre_slope < -0.5
                    pre_trend_category = 'weakening'      # Positive getting less positive (toward zero)
            elif pre_avg_momentum < 0:  # Negative momentum (Away team advantage)
                if pre_slope < -0.5:
                    pre_trend_category = 'strengthening'  # Negative getting more negative
                else:  # pre_slope > 0.5
                    pre_trend_category = 'weakening'      # Negative getting less negative (toward zero)
            else:
                pre_trend_category = 'neutral'
        
        if len(after_data) >= 2:
            z_after = np.polyfit(after_data['minute'], after_data['momentum'], 1)
            post_slope = z_after[0]
            p_after = np.poly1d(z_after)
            ss_res = np.sum((after_data['momentum'] - p_after(after_data['minute']))**2)
            ss_tot = np.sum((after_data['momentum'] - after_data['momentum'].mean())**2)
            post_r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            post_avg_momentum = after_data['momentum'].mean()
            post_momentum_magnitude = np.abs(post_avg_momentum)
        
        if pre_slope is not None and post_slope is not None:
            trend_change = post_slope - pre_slope
        
        windows.append({
            'match_id': match_id,
            'home_team': home_team,
            'away_team': away_team,
            'hydration_minute': hyd_minute,
            'pre_slope': pre_slope,
            'post_slope': post_slope,
            'pre_r2': pre_r2,
            'post_r2': post_r2,
            'trend_change': trend_change,
            'pre_avg_momentum': pre_avg_momentum,
            'post_avg_momentum': post_avg_momentum,
            'pre_momentum_magnitude': pre_momentum_magnitude,
            'post_momentum_magnitude': post_momentum_magnitude,
            'momentum_magnitude_change': post_momentum_magnitude - pre_momentum_magnitude if (pre_momentum_magnitude is not None and post_momentum_magnitude is not None) else None,
            'pre_trend_category': pre_trend_category,
            'before_data': before_data,
            'after_data': after_data,
            'full_window': window_data
        })
    
    return windows

# Apply to all matches
all_game_windows = []
for idx, row in merged_data.iterrows():
    game_windows = extract_hydration_windows(row)
    all_game_windows.extend(game_windows)

print(f'\nTotal hydration breaks analyzed: {len(all_game_windows)}')
print(f'Across {len(set([w["match_id"] for w in all_game_windows]))} unique matches')

# Display summary
if all_game_windows:
    summary_df = pd.DataFrame({
        'Match ID': [w['match_id'] for w in all_game_windows],
        'Teams': [f"{w['home_team']} vs {w['away_team']}" for w in all_game_windows],
        'Hydration Min': [w['hydration_minute'] for w in all_game_windows],
        'Pre Momentum': [f"{w['pre_avg_momentum']:.2f}" for w in all_game_windows],
        'Pre-Slope': [f"{w['pre_slope']:.3f}" for w in all_game_windows],
        'Post-Slope': [f"{w['post_slope']:.3f}" for w in all_game_windows],
        'Trend Category': [w['pre_trend_category'] for w in all_game_windows]
    })
    print('\nHydration Break Summary:')
    print(summary_df.to_string())

## Categorize Trends: Strengthening vs Weakening vs Neutral

In [ ]:
# Categorize windows by pre-treatment trend (CORRECTED logic)
# Strengthening: Momentum moving AWAY from zero (positive more positive, negative more negative)
# Weakening: Momentum moving TOWARD zero (positive less positive, negative less negative)
# Neutral: Momentum near zero at start

strengthening_trend = [w for w in all_game_windows if w['pre_trend_category'] == 'strengthening']
weakening_trend = [w for w in all_game_windows if w['pre_trend_category'] == 'weakening']
neutral_start = [w for w in all_game_windows if w['pre_trend_category'] == 'neutral']

print(f'Pre-treatment momentum STRENGTHENING (moving AWAY from zero): {len(strengthening_trend)} hydration breaks')
print(f'  - One team cementing/establishing advantage')
print(f'Pre-treatment momentum WEAKENING (moving TOWARD zero): {len(weakening_trend)} hydration breaks')
print(f'  - Advantage eroding, game becoming balanced')
print(f'Pre-treatment momentum NEUTRAL (near zero slope): {len(neutral_start)} hydration breaks')
print(f'  - Game balanced before hydration break')

print(f'\n' + '='*70)
print(f'STRENGTHENING trends - What happens during hydration break?')
print(f'(Expected: Advantage either CONTINUES to build OR REVERSES back toward neutral)')
if strengthening_trend:
    changes = [w['trend_change'] for w in strengthening_trend if w['trend_change'] is not None]
    if changes:
        print(f'  Mean trend change: {np.mean(changes):+.4f} (std: {np.std(changes):.4f})')
        continues = sum(1 for x in changes if (x > 0))  # Trend continues away from zero
        reverses = sum(1 for x in changes if (x < 0))    # Trend reverses toward zero
        print(f'  Advantage CONTINUES to build: {continues}/{len(changes)}')
        print(f'  Advantage REVERSES (hydration break helps trailing team): {reverses}/{len(changes)}')
        t_stat, p_val = stats.ttest_1samp(changes, 0)
        print(f'  T-test vs 0: t={t_stat:.4f}, p-value={p_val:.4f}')
        if p_val < 0.05:
            direction = 'ACCELERATES advantage' if np.mean(changes) > 0 else 'REVERSES advantage'
            print(f'  *** SIGNIFICANT: Hydration {direction} ***')
else:
    print(f'  No strengthening trends found')

print(f'\n' + '='*70)
print(f'WEAKENING trends - What happens during hydration break?')
print(f'(Expected: Game either CONTINUES toward balance OR REVERSES back to advantage)')
if weakening_trend:
    changes = [w['trend_change'] for w in weakening_trend if w['trend_change'] is not None]
    if changes:
        print(f'  Mean trend change: {np.mean(changes):+.4f} (std: {np.std(changes):.4f})')
        # For weakening trends: understand the momentum direction to interpret trend_change
        momentum_signs = [w['pre_avg_momentum'] > 0 for w in weakening_trend if w['trend_change'] is not None]
        continues_balance = sum(1 for w, momentum_pos in zip(weakening_trend, momentum_signs) 
                               if w['trend_change'] is not None and 
                               ((momentum_pos and w['trend_change'] < 0) or (not momentum_pos and w['trend_change'] > 0)))
        reverses_advantage = sum(1 for w, momentum_pos in zip(weakening_trend, momentum_signs)
                                if w['trend_change'] is not None and
                                ((momentum_pos and w['trend_change'] > 0) or (not momentum_pos and w['trend_change'] < 0)))
        print(f'  Game CONTINUES toward balance: {continues_balance}/{len(changes)}')
        print(f'  Advantage RE-ESTABLISHED (hydration restores lead): {reverses_advantage}/{len(changes)}')
        t_stat, p_val = stats.ttest_1samp(changes, 0)
        print(f'  T-test vs 0: t={t_stat:.4f}, p-value={p_val:.4f}')
        if p_val < 0.05:
            print(f'  *** SIGNIFICANT: Hydration affects weakening momentum trend ***')
else:
    print(f'  No weakening trends found')

print(f'\n' + '='*70)
print(f'NEUTRAL MOMENTUM - Does an advantage emerge after hydration?')
print(f'(Expected: Teams establish advantage OR momentum stays balanced)')
if neutral_start:
    mag_changes = [w['momentum_magnitude_change'] for w in neutral_start if w['momentum_magnitude_change'] is not None]
    if mag_changes:
        print(f'  Mean magnitude change: {np.mean(mag_changes):+.4f} (std: {np.std(mag_changes):.4f})')
        diverge_count = sum(1 for x in mag_changes if x > 0)
        converge_count = sum(1 for x in mag_changes if x < 0)
        print(f'  Momentum diverges (team establishes advantage): {diverge_count}/{len(mag_changes)}')
        print(f'  Momentum stays balanced: {converge_count}/{len(mag_changes)}')
        t_stat, p_val = stats.ttest_1samp(mag_changes, 0)
        print(f'  T-test vs 0: t={t_stat:.4f}, p-value={p_val:.4f}')
        if p_val < 0.05:
            direction = 'TEAMS ESTABLISH ADVANTAGE' if np.mean(mag_changes) > 0 else 'MAINTAINS BALANCE'
            print(f'  *** SIGNIFICANT: Hydration causes {direction} ***')
else:
    print(f'  No neutral momentum conditions found')

print(f'\n' + '='*70)

## Visualization 1: Overlay All Strengthening Pre-Treatment Trends

In [ ]:
# Create overlay plot for strengthening pre-treatment trends
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: All strengthening trends overlaid (pre-treatment)
for window in strengthening_trend:
    before_data = window['before_data']
    # Normalize x-axis: 0 = hydration time
    x_normalized = before_data['minute'] - window['hydration_minute']
    ax1.plot(x_normalized, before_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='green')
    
# Add mean trend line
if strengthening_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] 
                            for w in strengthening_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in strengthening_trend])
    
    z_mean = np.polyfit(all_x, all_y, 1)
    p_mean = np.poly1d(z_mean)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax1.plot(x_line, p_mean(x_line), 'g-', linewidth=3, label=f'Mean Slope: {z_mean[0]:.3f}')
    ax1.text(0.05, 0.95, 'Momentum MOVING AWAY FROM ZERO\n(positive getting more positive\nOR negative getting more negative)', 
             transform=ax1.transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

ax1.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Neutral Momentum')
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Hydration Break')
ax1.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax1.set_ylabel('Momentum\n(+ = Home, - = Away)', fontsize=11)
ax1.set_title(f'Pre-Treatment STRENGTHENING Trends (n={len(strengthening_trend)})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: All strengthening trends overlaid (post-treatment)
for window in strengthening_trend:
    after_data = window['after_data']
    x_normalized = after_data['minute'] - window['hydration_minute']
    ax2.plot(x_normalized, after_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='orange')

# Add mean trend line
if strengthening_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] 
                            for w in strengthening_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in strengthening_trend])
    
    if len(all_x) > 1:
        z_mean = np.polyfit(all_x, all_y, 1)
        p_mean = np.poly1d(z_mean)
        x_line = np.linspace(0, all_x.max(), 100)
        ax2.plot(x_line, p_mean(x_line), 'orange', linewidth=3, label=f'Mean Slope: {z_mean[0]:.3f}')
        ax2.text(0.05, 0.95, 'After hydration: does advantage\ncontinue building or reverse?', 
                 transform=ax2.transAxes, fontsize=10, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

ax2.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Neutral Momentum')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Hydration Break')
ax2.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax2.set_ylabel('Momentum\n(+ = Home, - = Away)', fontsize=11)
ax2.set_title(f'Post-Treatment STRENGTHENING Trends (n={len(strengthening_trend)})', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('corrected_strengthening_trends_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: corrected_strengthening_trends_overlay.png')

## Visualization 2: Overlay All Weakening Pre-Treatment Trends

In [ ]:
# Create overlay plot for weakening pre-treatment trends
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: All weakening trends overlaid (pre-treatment)
for window in weakening_trend:
    before_data = window['before_data']
    x_normalized = before_data['minute'] - window['hydration_minute']
    ax1.plot(x_normalized, before_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='blue')

# Add mean trend line
if weakening_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] 
                            for w in weakening_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in weakening_trend])
    
    z_mean = np.polyfit(all_x, all_y, 1)
    p_mean = np.poly1d(z_mean)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax1.plot(x_line, p_mean(x_line), 'b-', linewidth=3, label=f'Mean Slope: {z_mean[0]:.3f}')
    ax1.text(0.05, 0.95, 'Momentum MOVING TOWARD ZERO\n(positive getting less positive\nOR negative getting less negative)', 
             transform=ax1.transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

ax1.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Neutral Momentum')
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Hydration Break')
ax1.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax1.set_ylabel('Momentum\n(+ = Home, - = Away)', fontsize=11)
ax1.set_title(f'Pre-Treatment WEAKENING Trends (n={len(weakening_trend)})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: All weakening trends overlaid (post-treatment)
for window in weakening_trend:
    after_data = window['after_data']
    x_normalized = after_data['minute'] - window['hydration_minute']
    ax2.plot(x_normalized, after_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='red')

# Add mean trend line
if weakening_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] 
                            for w in weakening_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in weakening_trend])
    
    if len(all_x) > 1:
        z_mean = np.polyfit(all_x, all_y, 1)
        p_mean = np.poly1d(z_mean)
        x_line = np.linspace(0, all_x.max(), 100)
        ax2.plot(x_line, p_mean(x_line), 'r-', linewidth=3, label=f'Mean Slope: {z_mean[0]:.3f}')
        ax2.text(0.05, 0.95, 'After hydration: does balance continue\nor does advantage re-establish?', 
                 transform=ax2.transAxes, fontsize=10, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

ax2.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Neutral Momentum')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Hydration Break')
ax2.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax2.set_ylabel('Momentum\n(+ = Home, - = Away)', fontsize=11)
ax2.set_title(f'Post-Treatment WEAKENING Trends (n={len(weakening_trend)})', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('corrected_weakening_trends_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: corrected_weakening_trends_overlay.png')

## Visualization 3: Neutral Start Conditions - How Much Does Momentum Diverge?

In [ ]:
# For neutral starts, we care about POST-treatment momentum magnitude
# Do teams establish momentum after hydration break when they didn't have it before?

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Pre-treatment (all should be near zero)
for window in neutral_start:
    before_data = window['before_data']
    x_normalized = before_data['minute'] - window['hydration_minute']
    ax1.plot(x_normalized, before_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='purple')
    ax1.scatter(x_normalized, before_data['momentum'], alpha=0.2, s=20, color='purple')

# Add mean line
if neutral_start:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] 
                            for w in neutral_start])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in neutral_start])
    
    z_mean = np.polyfit(all_x, all_y, 1)
    p_mean = np.poly1d(z_mean)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax1.plot(x_line, p_mean(x_line), 'purple', linewidth=3, label=f'Mean Slope: {z_mean[0]:.3f}')
    ax1.fill_between(x_line, -2, 2, alpha=0.1, color='gray', label='Neutral Zone (-2 to +2)')
    ax1.text(0.05, 0.95, 'Game is balanced before\nhydration break', 
             transform=ax1.transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.7))

ax1.axhline(y=0, color='black', linestyle='-', linewidth=2, alpha=0.7, label='Perfect Neutral')
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Hydration Break')
ax1.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax1.set_ylabel('Momentum\n(+ = Home, - = Away)', fontsize=11)
ax1.set_title(f'Pre-Treatment NEUTRAL Momentum (n={len(neutral_start)})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-15, 15)

# Plot 2: Post-treatment (shows divergence from neutral)
for window in neutral_start:
    after_data = window['after_data']
    x_normalized = after_data['minute'] - window['hydration_minute']
    ax2.plot(x_normalized, after_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='brown')
    ax2.scatter(x_normalized, after_data['momentum'], alpha=0.2, s=20, color='brown')

# Add mean line
if neutral_start:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] 
                            for w in neutral_start])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in neutral_start])
    
    if len(all_x) > 1:
        z_mean = np.polyfit(all_x, all_y, 1)
        p_mean = np.poly1d(z_mean)
        x_line = np.linspace(0, all_x.max(), 100)
        ax2.plot(x_line, p_mean(x_line), 'brown', linewidth=3, label=f'Mean Slope: {z_mean[0]:.3f}')
        ax2.fill_between(x_line, -2, 2, alpha=0.1, color='gray', label='Neutral Zone (-2 to +2)')
        ax2.text(0.05, 0.95, 'Does a team establish\nadvantage after break?', 
                 transform=ax2.transAxes, fontsize=10, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

ax2.axhline(y=0, color='black', linestyle='-', linewidth=2, alpha=0.7, label='Perfect Neutral')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Hydration Break')
ax2.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax2.set_ylabel('Momentum\n(+ = Home, - = Away)', fontsize=11)
ax2.set_title(f'Post-Treatment NEUTRAL Start (shows divergence from neutral)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-15, 15)

plt.tight_layout()
plt.savefig('corrected_neutral_start_divergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: corrected_neutral_start_divergence.png')

## Visualization 4: Three-Way Comparison Grid

In [ ]:
# Side-by-side comparison: before and after for all three trend types
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Row 1: Strengthening trends
ax = fig.add_subplot(gs[0, 0])
for window in strengthening_trend:
    before_data = window['before_data']
    x_norm = before_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, before_data['momentum'], alpha=0.3, linewidth=1, color='green')

if strengthening_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] for w in strengthening_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in strengthening_trend])
    z = np.polyfit(all_x, all_y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax.plot(x_line, p(x_line), 'g-', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title('STRENGTHENING Pre-Treatment\n(away from zero)', fontsize=11, fontweight='bold')
ax.set_ylabel('Momentum', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[0, 1])
for window in strengthening_trend:
    after_data = window['after_data']
    x_norm = after_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, after_data['momentum'], alpha=0.3, linewidth=1, color='orange')

if strengthening_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] for w in strengthening_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in strengthening_trend])
    if len(all_x) > 1:
        z = np.polyfit(all_x, all_y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(0, all_x.max(), 100)
        ax.plot(x_line, p(x_line), 'orange', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
ax.set_title('STRENGTHENING Post-Treatment', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Row 2: Weakening trends
ax = fig.add_subplot(gs[1, 0])
for window in weakening_trend:
    before_data = window['before_data']
    x_norm = before_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, before_data['momentum'], alpha=0.3, linewidth=1, color='blue')

if weakening_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] for w in weakening_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in weakening_trend])
    z = np.polyfit(all_x, all_y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax.plot(x_line, p(x_line), 'b-', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title('WEAKENING Pre-Treatment\n(toward zero)', fontsize=11, fontweight='bold')
ax.set_ylabel('Momentum', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[1, 1])
for window in weakening_trend:
    after_data = window['after_data']
    x_norm = after_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, after_data['momentum'], alpha=0.3, linewidth=1, color='red')

if weakening_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] for w in weakening_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in weakening_trend])
    if len(all_x) > 1:
        z = np.polyfit(all_x, all_y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(0, all_x.max(), 100)
        ax.plot(x_line, p(x_line), 'r-', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
ax.set_title('WEAKENING Post-Treatment', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Row 3: Neutral start
ax = fig.add_subplot(gs[2, 0])
for window in neutral_start:
    before_data = window['before_data']
    x_norm = before_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, before_data['momentum'], alpha=0.3, linewidth=1, color='purple')

if neutral_start:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] for w in neutral_start])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in neutral_start])
    z = np.polyfit(all_x, all_y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax.plot(x_line, p(x_line), 'purple', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axhline(y=0, color='black', linestyle='-', linewidth=2, alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title('NEUTRAL Pre-Treatment\n(balanced game)', fontsize=11, fontweight='bold')
ax.set_xlabel('Minutes from Hydration Break', fontsize=10)
ax.set_ylabel('Momentum', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-10, 10)

ax = fig.add_subplot(gs[2, 1])
for window in neutral_start:
    after_data = window['after_data']
    x_norm = after_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, after_data['momentum'], alpha=0.3, linewidth=1, color='brown')

if neutral_start:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] for w in neutral_start])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in neutral_start])
    if len(all_x) > 1:
        z = np.polyfit(all_x, all_y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(0, all_x.max(), 100)
        ax.plot(x_line, p(x_line), 'brown', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axhline(y=0, color='black', linestyle='-', linewidth=2, alpha=0.7)
ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
ax.set_title('NEUTRAL Post-Treatment\n(advantage emergence?)', fontsize=11, fontweight='bold')
ax.set_xlabel('Minutes from Hydration Break', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-10, 10)

plt.suptitle('Hydration Break Effects on Momentum by Pre-Treatment Condition\n(CORRECTED: Strengthening = away from zero | Weakening = toward zero)', fontsize=14, fontweight='bold', y=0.995)
plt.savefig('corrected_three_way_trend_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: corrected_three_way_trend_comparison.png')

## Summary Statistics and Key Findings

In [ ]:
print("\n" + "="*80)
print("CORRECTED RDD VISUAL ANALYSIS - COMPREHENSIVE SUMMARY REPORT")
print("="*80)
print("\nKEY DEFINITIONS (CORRECTED):")
print("  STRENGTHENING: Momentum moving AWAY from zero")
print("    • Positive momentum getting MORE positive (home team advantage expanding)")
print("    • Negative momentum getting MORE negative (away team advantage expanding)")
print("  WEAKENING: Momentum moving TOWARD zero")
print("    • Positive momentum getting LESS positive (home advantage eroding)")
print("    • Negative momentum getting LESS negative (away advantage eroding)")
print("  NEUTRAL: Momentum near zero (balanced game)")

print(f"\n" + "="*80)
print(f"TOTAL DATA:")
print(f"  Hydration breaks analyzed: {len(all_game_windows)}")
print(f"  Unique matches: {len(set([w['match_id'] for w in all_game_windows]))}")

print(f"\nCATEGORIZATION (by pre-treatment momentum direction):")
print(f"  STRENGTHENING (moving away from zero): {len(strengthening_trend)}")
print(f"  WEAKENING (moving toward zero): {len(weakening_trend)}")
print(f"  NEUTRAL (near zero): {len(neutral_start)}")

print(f"\n" + "="*80)
print(f"STRENGTHENING MOMENTUM - What happens during hydration break?")
print(f"(Advantage is already being established before break)")
print(f"="*80)
if strengthening_trend:
    s_changes = [w['trend_change'] for w in strengthening_trend if w['trend_change'] is not None]
    if s_changes:
        print(f"  Mean trend change: {np.mean(s_changes):+.4f} (std: {np.std(s_changes):.4f})")
        print(f"  Median trend change: {np.median(s_changes):+.4f}")
        accel_count = sum(1 for x in s_changes if x > 0)
        decel_count = sum(1 for x in s_changes if x < 0)
        print(f"  Advantage CONTINUES to build (positive trend change): {accel_count}/{len(s_changes)}")
        print(f"  Advantage PAUSES/REVERSES (negative trend change): {decel_count}/{len(s_changes)}")
        t_stat, p_val = stats.ttest_1samp(s_changes, 0)
        print(f"  T-test vs 0: t={t_stat:.4f}, p-value={p_val:.4f}")
        if p_val < 0.05:
            if np.mean(s_changes) > 0:
                print(f"  *** SIGNIFICANT: Hydration ACCELERATES strengthening momentum ***")
            else:
                print(f"  *** SIGNIFICANT: Hydration DISRUPTS strengthening momentum ***")
else:
    print(f"  No strengthening trends found")

print(f"\n" + "="*80)
print(f"WEAKENING MOMENTUM - What happens during hydration break?")
print(f"(Advantage is eroding before break)")
print(f"="*80)
if weakening_trend:
    w_changes = [w['trend_change'] for w in weakening_trend if w['trend_change'] is not None]
    if w_changes:
        print(f"  Mean trend change: {np.mean(w_changes):+.4f} (std: {np.std(w_changes):.4f})")
        print(f"  Median trend change: {np.median(w_changes):+.4f}")
        # For weakening trends, understand what trend_change means based on momentum sign
        momentum_signs = [w['pre_avg_momentum'] > 0 for w in weakening_trend if w['trend_change'] is not None]
        continues_balance = sum(1 for w, momentum_pos in zip(weakening_trend, momentum_signs) 
                               if w['trend_change'] is not None and 
                               ((momentum_pos and w['trend_change'] < 0) or (not momentum_pos and w['trend_change'] > 0)))
        reverses_advantage = sum(1 for w, momentum_pos in zip(weakening_trend, momentum_signs)
                                if w['trend_change'] is not None and
                                ((momentum_pos and w['trend_change'] > 0) or (not momentum_pos and w['trend_change'] < 0)))
        print(f"  Erosion CONTINUES (trend toward zero accelerates): {continues_balance}/{len(w_changes)}")
        print(f"  Advantage RE-ESTABLISHED (trend reverses away from zero): {reverses_advantage}/{len(w_changes)}")
        t_stat, p_val = stats.ttest_1samp(w_changes, 0)
        print(f"  T-test vs 0: t={t_stat:.4f}, p-value={p_val:.4f}")
        if p_val < 0.05:
            if np.mean(w_changes) < 0:
                print(f"  *** SIGNIFICANT: Hydration ACCELERATES momentum erosion ***")
            else:
                print(f"  *** SIGNIFICANT: Hydration REVERSES momentum erosion (restores lead) ***")
else:
    print(f"  No weakening trends found")

print(f"\n" + "="*80)
print(f"NEUTRAL MOMENTUM - Does a team establish advantage after hydration?")
print(f"(Game is balanced before break)")
print(f"="*80)
if neutral_start:
    n_mag_changes = [w['momentum_magnitude_change'] for w in neutral_start if w['momentum_magnitude_change'] is not None]
    if n_mag_changes:
        print(f"  Mean magnitude change: {np.mean(n_mag_changes):+.4f} (std: {np.std(n_mag_changes):.4f})")
        print(f"  Median magnitude change: {np.median(n_mag_changes):+.4f}")
        diverge_count = sum(1 for x in n_mag_changes if x > 0)
        converge_count = sum(1 for x in n_mag_changes if x < 0)
        print(f"  Team ESTABLISHES advantage (momentum away from zero): {diverge_count}/{len(n_mag_changes)}")
        print(f"  Game STAYS BALANCED (momentum toward zero): {converge_count}/{len(n_mag_changes)}")
        t_stat, p_val = stats.ttest_1samp(n_mag_changes, 0)
        print(f"  T-test vs 0: t={t_stat:.4f}, p-value={p_val:.4f}")
        if p_val < 0.05:
            if np.mean(n_mag_changes) > 0:
                print(f"  *** SIGNIFICANT: Hydration ENABLES a team to establish advantage ***")
            else:
                print(f"  *** SIGNIFICANT: Hydration MAINTAINS balance in neutral situations ***")
else:
    print(f"  No neutral momentum conditions found")

print(f"\n" + "="*80)